# ABSA — Dataset Build & Exploratory Analysis

Runs the full data pipeline and renders the EDA figures. Works in Google Colab
and locally; no GPU required (this notebook is CPU-only — training lives in
`notebooks/absa_training.ipynb`).

**The logic is not in this notebook.** It lives in `ml/preprocessing/` and
`ml/eda.py`, which this notebook imports. That keeps the pipeline diffable in
git, testable in CI, and identical whether it runs here or from the CLI.

Pipeline:

```
download → parse → clean → transform → dedup/split (leakage-asserted) → EDA
```

## 1. Get the code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_NAME = "absa-platform"
# Point this at your fork/remote when running in Colab.
REPO_URL = os.environ.get("ABSA_REPO_URL", "https://github.com/YOUR_USERNAME/absa-platform.git")

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone -q $REPO_URL $REPO_NAME
    else:
        !cd $REPO_NAME && git pull -q
    REPO_ROOT = Path(REPO_NAME).resolve()
else:
    # Local: the notebook sits in <repo>/notebooks/
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f"Colab: {IN_COLAB}")
print(f"Repo:  {REPO_ROOT}")

## 2. Dependencies

Colab already ships pandas/numpy/sklearn/matplotlib, so this is usually a no-op.

In [ ]:
if IN_COLAB:
    !pip install -q -r requirements.txt

import matplotlib, pandas as pd, sklearn, yaml
print(f"pandas {pd.__version__} | sklearn {sklearn.__version__} | matplotlib {matplotlib.__version__}")

## 3. Download the dataset

**M-ABSA** (Wu et al., EMNLP 2025) — English `phone` + `laptop` splits.
The upstream repo ships no LICENSE, so the data is fetched rather than
redistributed; `CITATION.bib` is written alongside it.

In [ ]:
!python scripts/download_data.py

## 4. Inspect the raw format

One review per line: `text####[[term, category, polarity], ...]`

In [ ]:
raw = Path("data/raw/mabsa/phone/train.txt").read_text(encoding="utf-8").splitlines()
print(f"phone/train.txt — {len(raw)} lines\n")
for line in raw[:2]:
    text, _, triplets = line.partition("####")
    print(f"TEXT:     {text[:150]}")
    print(f"TRIPLETS: {triplets[:220]}\n")

## 5. Build the datasets

Cleans conservatively (negations, casing and punctuation are **kept** — they carry
sentiment), collapses 86 + 108 upstream categories onto 12 aspects, majority-votes
polarity per `(review, aspect)`, then splits **grouped by review id and by
normalised text**.

Splitting at row level would scatter sentences from one review across train and
test — the usual way an ABSA project ends up reporting 99%. The build asserts
this cannot happen and fails loudly if it does.

In [ ]:
!python scripts/build_dataset.py

## 6. Exploratory analysis

In [ ]:
from ml.eda import compute_stats, load_processed, render_all

asc = load_processed(Path("data/processed"))
stats = compute_stats(asc)
asc.head()

In [ ]:
!python scripts/run_eda.py

### Figures

In [ ]:
from IPython.display import Image, display

for name in (
    "aspect_distribution.png",
    "polarity_by_aspect.png",
    "review_length.png",
    "aspects_per_review.png",
):
    display(Image(filename=f"docs/figures/{name}"))

## 7. What the EDA tells us before any training

1. **Class imbalance is the headline risk.** `neutral` is ~5% of pairs. Accuracy
   would be flattered by a model that never predicts it, so **macro F1 and
   per-class metrics are the selection criteria**, not accuracy.
2. **Aspect frequency is very skewed.** `overall` alone is ~37% of pairs, while
   `audio` has ~136. Expect weak per-class scores on the tail, and report them.
3. **Reviews are short** — median 16 words, p99 107. `max_length=128` covers
   ~99.6% of reviews, so truncation is not a meaningful source of error.
4. **~1,245 reviews mention 2+ aspects.** That is precisely why splitting is
   grouped by review.
5. **Negativity varies sharply by aspect** — `audio` ~53% negative vs `price`
   ~9%. A model that ignores the aspect and reads only overall tone will fail
   in a visible, diagnosable way.

Next: `notebooks/absa_training.ipynb` — baselines, then transformers, on a
Colab GPU.